# Multi-Stage Retrieval for BDI-II Symptom Ranking in Social Media: eRisk2025

## Students Name and Email ID
Name: Kulsoom

Surname: Zaidi

e-mail: kulsoom.zaidi@studio.unibo.it

Name: Alina

Surname: Baig

e-mail: alina.baig2@studio.unibo.it



##**CLEF eRisk 2025: Early risk prediction on the Internet**

eRisk (Early Risk Prediction on the Internet) is an annual lab associated with the CLEF conference that evaluates methods for detecting personal risk indicators from online text. In 2025, the lab expanded its scope toward depression-related challenges, adding contextual and conversational tasks while continuing a core ranking task focused on depressive symptoms.

##Task 1: Search for Symptoms of Depression

> Task 1 — Search for Symptoms of Depression is a retrieval and ranking task where systems are evaluated on their ability to identify and rank sentences from a large text collection according to their relevance to specific depressive symptoms.

> The symptoms to be detected are defined by the Beck Depression Inventory‑II (BDI‑II) — a standard clinical questionnaire with 21 symptom items (e.g., sadness, loss of energy, sleep changes). For each symptom, the system must score and rank up to 1,000 sentences from a corpus where higher scores indicate stronger relevance to that symptom.





##Setup

In [ ]:
import os          # For interacting with the operating system (file paths, directories, etc.)
import re          # For regular expressions, useful in text processing
import csv         # For reading and writing CSV files
import sqlite3     # For interacting with SQLite databases
import shutil      # For high-level file operations (copying, moving, removing directories)
import json        # For working with JSON data
import zipfile     # For handling ZIP archives

import numpy as np                 # Numerical computing library, often used for arrays and math operations
from tqdm import tqdm              # Progress bar utility, useful for loops and iterations
from collections import defaultdict  # Dictionary subclass that provides default values for missing keys


!pip install transformers -q      # Hugging Face Transformers library for NLP models
!pip install accelerate -U -q    # Accelerate library to speed up model training on GPUs
!pip install evaluate -q          # Evaluation library for NLP metrics
!pip install -U bitsandbytes -q  # Efficient 8-bit model training for large language models

## GPU memory

> Clears unused GPU memory, checks if a CUDA-enabled GPU is available using PyTorch, and if so, prints the amount of free GPU memory in gigabytes.




In [ ]:
import torch
torch.cuda.empty_cache()
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

GPU available: True
GPU memory free: 15.5 GB


## HuggingFace Login


In [ ]:
!hf auth login

A new version of huggingface_hub (1.9.0) is available! You are using version 1.8.0.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: n
Token is valid (permission: read).
The token `z` has been saved to /root/.cache/huggingface/stored_token

##Pipeline for eRisk 2025 Task 1

1. BM25 Baseline Retrieval
2. Semantic Reranking using Sentence Embeddings
3. Hybrid Scoring (BM25 + Embeddings)
4. LLM‑Based Relevance Scoring with Mistral

## Streaming Parser



> This code defines a streaming parser for .trec files, which are structured with `<DOC>`, `<DOCNO>`, `<TEXT>`, `<PRE>`, and `<POST>` tags. It reads the file line by line, extracts document metadata and content, and yields each document as a dictionary containing:




*   `doc_id` – full document identifier

*   `user_id` – user identifier (from doc_id)
*   `post_idx` – post index


*   `chunk_idx` – chunk index

* `pre, text, post` – content sections

The parser handles multi-line content and missing tags.

In [ ]:
# Regular expressions to extract content from TREC-style tags
RE_DOCNO = re.compile(r"<DOCNO>(.*?)</DOCNO>")
RE_TEXT  = re.compile(r"<TEXT>(.*?)</TEXT>")
RE_PRE   = re.compile(r"<PRE>(.*?)</PRE>")
RE_POST  = re.compile(r"<POST>(.*?)</POST>")


def parse_trec_file_stream(filepath: str):

  """
    Stream and parse a TREC-formatted file (.trec) document by document.

    Yields:
        dict with keys:
            - doc_id    : full DOCNO string
            - user_id   : user identifier (first part of DOCNO)
            - post_idx  : post index (second part of DOCNO, if exists)
            - chunk_idx : chunk index (third part of DOCNO, if exists)
            - pre       : optional PRE text
            - text      : main TEXT content
            - post      : optional POST text
    """
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        inside  = False
        docno   = text = pre = post = current = None
        for line in f:
            stripped = line.strip()
            # Start of a new document
            if "<DOC>" in stripped:
                inside  = True
                docno   = text = pre = post = current = None
                continue

            # End of a document → yield the parsed dictionary
            if "</DOC>" in stripped:
                if inside and docno and text is not None:
                    parts = docno.split("_")
                    yield {
                        "doc_id"    : docno,
                        "user_id"   : parts[0] if len(parts) >= 1 else None,
                        "post_idx"  : int(parts[1]) if len(parts) >= 2 else None,
                        "chunk_idx" : int(parts[2]) if len(parts) >= 3 else None,
                        "pre"       : pre  or "",
                        "text"      : text,
                        "post"      : post or "",
                    }
                inside  = False
                current = None
                continue
            if not inside:
                continue

            # Extract DOCNO
            if "<DOCNO>" in stripped:
                docno   = stripped.replace("<DOCNO>", "").replace("</DOCNO>", "").strip()
                current = None
                continue

            # Extract TEXT (multi-line supported)
            if "<TEXT>" in stripped:
                text    = stripped.replace("<TEXT>", "").replace("</TEXT>", "").strip()
                current = "text" if "</TEXT>" not in stripped else None
                continue
            if "</TEXT>" in stripped:
                if text is not None and current == "text":
                    text += " " + stripped.replace("</TEXT>", "").strip()
                current = None
                continue

            # Extract PRE (multi-line supported)
            if "<PRE>" in stripped:
                pre     = stripped.replace("<PRE>", "").replace("</PRE>", "").strip()
                current = "pre" if "</PRE>" not in stripped else None
                continue
            if "</PRE>" in stripped:
                if pre is not None and current == "pre":
                    pre += " " + stripped.replace("</PRE>", "").strip()
                current = None
                continue

            # Extract POST (multi-line supported)
            if "<POST>" in stripped:
                post    = stripped.replace("<POST>", "").replace("</POST>", "").strip()
                current = "post" if "</POST>" not in stripped else None
                continue
            if "</POST>" in stripped:
                if post is not None and current == "post":
                    post += " " + stripped.replace("</POST>", "").strip()
                current = None
                continue

            # Append additional lines for multi-line TEXT, PRE, POST
            if current == "text" and text is not None:
                text += " " + stripped
            elif current == "pre" and pre is not None:
                pre  += " " + stripped
            elif current == "post" and post is not None:
                post += " " + stripped

# Test the streaming parser on the first .trec file in the dataset directory
test_file = os.path.join(DATASET_DIR, sorted(
    [f for f in os.listdir(DATASET_DIR) if f.endswith(".trec")],
    key=lambda f: int(re.search(r"\d+", f).group())
)[0])

# Count total documents parsed
count = 0
for doc in parse_trec_file_stream(test_file):
    count += 1
print(f"Streaming parser ready — {count} docs from test file")

Streaming parser ready — 703 docs from test file


## Preprocessing
> This code defines a text preprocessing function to clean and normalize raw text. It performs the following steps:



* Remove URLs – strips links starting with http or www.
* Simplify markdown links – converts (url) into text.
* Remove special characters – removes symbols like *, _, ~, `, #, > which are common in markdown or formatting.
* Normalize whitespace – collapses multiple spaces into a single space and trims leading/trailing spaces.
* Lowercase – converts all text to lowercase for consistency.

In [ ]:
def preprocess(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    text = re.sub(r"[*_~`#>]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

print("Preprocess ready.")

Preprocess ready.


## Load qrels

It processes the qrels CSV, which contains user posts labeled for relevance to early risk detection. It performs the following steps:

* Read the CSV – Loads the CSV containing query (user ID), doc_id (post ID), and relevant (True/False) columns.
* Parse rows – Converts each row into a structured dictionary with boolean relevance.
* Label analysis – Counts how many posts are relevant vs. non-relevant and prints percentages and imbalance ratio.

In [ ]:
from collections import Counter

#Step 1: Read qrels CSV and store rows
qrels_rows = []
with open(QRELS_CSV, "r", encoding="utf-8") as f:
    reader  = csv.DictReader(f)
    headers = reader.fieldnames
    print(f"CSV columns: {headers}\n")
    for row in reader:
      # Convert 'relevant' field to Boolean and store query, doc_id, relevance
        qrels_rows.append({
            "query"   : row["query"],
            "doc_id"  : row["doc_id"],
            "relevant": row["relevant"].strip().lower() == "true"
        })

print(f"Total rows in qrels CSV : {len(qrels_rows):,}")

label_counts = Counter(row["relevant"] for row in qrels_rows)
total        = len(qrels_rows)
true_count   = label_counts.get(True,  0)
false_count  = label_counts.get(False, 0)

print(f"\n=== Label Distribution ===")
print(f"  Relevant (True)     : {true_count:,}  ({100*true_count/total:.1f}%)" if total > 0 else "  No data")
print(f"  Not Relevant (False): {false_count:,}  ({100*false_count/total:.1f}%)" if total > 0 else "  No data")
if true_count > 0 and false_count > 0:
    print(f"  Imbalance ratio     : {false_count/true_count:.1f}x more negatives")
# Step 3: Query distribution
query_counts = Counter(row["query"] for row in qrels_rows)
print(f"\n=== Query Distribution ===")
for q, count in sorted(query_counts.items(), key=lambda x: int(x[0])):
    print(f"  Query {q:>2} : {count:,} rows")

CSV columns: ['query', 'doc_id', 'relevant']

Total rows in qrels CSV : 11,042

=== Label Distribution ===
  Relevant (True)     : 6,117  (55.4%)
  Not Relevant (False): 4,925  (44.6%)
  Imbalance ratio     : 0.8x more negatives

=== Query Distribution ===
  Query  1 : 581 rows
  Query  2 : 552 rows
  Query  3 : 536 rows
  Query  4 : 522 rows
  Query  5 : 400 rows
  Query  6 : 553 rows
  Query  7 : 474 rows
  Query  8 : 534 rows
  Query  9 : 517 rows
  Query 10 : 547 rows
  Query 11 : 593 rows
  Query 12 : 553 rows
  Query 13 : 584 rows
  Query 14 : 424 rows
  Query 15 : 491 rows
  Query 16 : 569 rows
  Query 17 : 540 rows
  Query 18 : 548 rows
  Query 19 : 428 rows
  Query 20 : 566 rows
  Query 21 : 530 rows


## Build qrels Lookup + Join with Posts

This code joins relevent labels with the actual user posts from the .trec files, creating per-user datasets for modeling. Steps:

* Build lookup table – Creates a dictionary qrels_lookup mapping (doc_id, query) pairs to relevance labels.
* Map documents to queries – Uses doc_to_queries to track which users (queries) each document belongs to.
* Prepare per-query datasets – Iterates over all .trec files, parses documents, and collects only those present in the qrels, storing them in query_datasets by query/user.
* Track progress and missing documents – Uses tqdm to show progress and identifies any labeled documents not found in the dataset.
* Summarize dataset – Prints the number of labeled documents found/missing and per-query statistics, including total documents, positives, and negatives.

In [ ]:
# Step 1: Build a quick lookup for (doc_id, query) → relevance
qrels_lookup = {}
for row in qrels_rows:
    key = (row["doc_id"], row["query"])
    qrels_lookup[key] = row["relevant"]

print(f"Total unique (doc_id, query) pairs: {len(qrels_lookup):,}")

# Step 2: Map each doc_id to all queries it appears in
doc_to_queries = defaultdict(list)
for (doc_id, query), label in qrels_lookup.items():
    doc_to_queries[doc_id].append((query, label))

# Prepare for joining with actual post content
query_datasets = defaultdict(list)
target_doc_ids = set(doc_id for (doc_id, query) in qrels_lookup.keys())
print(f"Unique doc_ids to find: {len(target_doc_ids):,}")

# Step 3: Stream through TREC files and join labels with post content
found_count = 0
for filename in tqdm(
    sorted(trec_files, key=lambda f: int(re.search(r"\d+", f).group())),
    desc="Joining labels with posts"
):
    filepath = os.path.join(DATASET_DIR, filename)
    for doc in parse_trec_file_stream(filepath):
        if doc["doc_id"] not in target_doc_ids:
            continue
        found_count += 1

        # For each query the doc belongs to, add labeled dataset entry
        for query, label in doc_to_queries[doc["doc_id"]]:
            query_datasets[query].append({
                "doc_id" : doc["doc_id"],
                "user_id": doc["user_id"],
                "text"   : doc["text"],
                "pre"    : doc["pre"],
                "post"   : doc["post"],
                "label"  : label,
                "query"  : query,
            })

found_ids    = set(doc["doc_id"] for docs in query_datasets.values() for doc in docs)
missing_docs = target_doc_ids - found_ids
print(f"\nJoin complete!")
print(f"   Labeled docs found   : {found_count:,}")
print(f"   Labeled docs missing : {len(missing_docs):,}")

# Step 5: Summarize dataset sizes per query
print(f"\n=== Per-Query Dataset Sizes ===")
for query in sorted(query_datasets.keys(), key=lambda x: int(x)):
    docs = query_datasets[query]
    pos  = sum(1 for d in docs if d["label"] == True)
    neg  = sum(1 for d in docs if d["label"] == False)
    print(f"  Query {query:>2} : {len(docs):,} docs  |  + {pos} positive  |  - {neg} negative")

Total unique (doc_id, query) pairs: 11,042
Unique doc_ids to find: 10,383


Joining labels with posts: 100%|██████████| 6300/6300 [01:50<00:00, 57.10it/s]


Join complete!
   Labeled docs found   : 10,383
   Labeled docs missing : 0

=== Per-Query Dataset Sizes ===
  Query  1 : 581 docs  |  + 296 positive  |  - 285 negative
  Query  2 : 552 docs  |  + 345 positive  |  - 207 negative
  Query  3 : 536 docs  |  + 283 positive  |  - 253 negative
  Query  4 : 522 docs  |  + 244 positive  |  - 278 negative
  Query  5 : 400 docs  |  + 227 positive  |  - 173 negative
  Query  6 : 553 docs  |  + 111 positive  |  - 442 negative
  Query  7 : 474 docs  |  + 290 positive  |  - 184 negative
  Query  8 : 534 docs  |  + 259 positive  |  - 275 negative
  Query  9 : 517 docs  |  + 377 positive  |  - 140 negative
  Query 10 : 547 docs  |  + 359 positive  |  - 188 negative
  Query 11 : 593 docs  |  + 338 positive  |  - 255 negative
  Query 12 : 553 docs  |  + 229 positive  |  - 324 negative
  Query 13 : 584 docs  |  + 139 positive  |  - 445 negative
  Query 14 : 424 docs  |  + 249 positive  |  - 175 negative
  Query 15 : 491 docs  |  + 273 positive  |  - 218

## Build labeled_docs Lookup

This code creates a quick lookup dictionary of all labeled documents by their doc_id, storing only the main text content. Steps:

* Iterate over per-query datasets – Loops through all documents in query_datasets.
* Build labeled_docs dictionary – Maps each doc_id to its corresponding text for fast access.
* Print summary – Shows the total number of unique labeled documents and a sample snippet of one document.

In [ ]:
# Build a dictionary of all labeled documents keyed by doc_id
labeled_docs = {}

# Iterate over all per-query datasets
for docs in query_datasets.values():
    for d in docs:
        labeled_docs[d["doc_id"]] = d.get("text", "")

# Print total number of unique labeled documents
print(f"labeled_docs built — {len(labeled_docs):,} unique doc_ids")

# Preview a sample document
sample_key = next(iter(labeled_docs))
print(f"Sample: {sample_key!r}: {labeled_docs[sample_key][:80]!r}")

labeled_docs built — 10,383 unique doc_ids
Sample: 'PgZVTC_76_0': 'My SO snored so loud one night that it scared me out of my sleep.'


## Build SQLite FTS5 Index

This code handles building and connecting to a full-text search index.

* get_connection – Connects to the local SQLite DB, copying from Drive if missing, and verifies the posts table.
* build_index – Creates/rebuilds an FTS5 index from .trec files, combining and preprocessing pre, text, and post fields, skipping empty documents, and storing them in posts (for search) and posts_lookup (for fast retrieval).
* Usage – With FORCE_REBUILD=False, the existing index is reused; conn and c are ready for querying.

In [ ]:
def get_connection(drive_db: str, local_db: str) -> sqlite3.Connection:
  # If local database does not exist, try copying from Drive
    if not os.path.exists(local_db):
        if not os.path.exists(drive_db):
            raise FileNotFoundError(f"No DB found at {drive_db} — run build_index first.")
        print("Copying DB from Drive to local storage...")
        shutil.copy(drive_db, local_db)
        print("Copy complete!")
    else:
        print("Local DB already exists — connecting directly.")

    # Connect to the local SQLite database
    conn = sqlite3.connect(local_db, timeout=30)
    c    = conn.cursor()

    # Check if the 'posts' table exists
    c.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='posts'")
    if c.fetchone() is None:
        raise RuntimeError("DB connected but 'posts' table not found — rebuild index.")
    c.execute("SELECT COUNT(*) FROM posts")
    count = c.fetchone()[0]
    print(f"Connected! Index has {count:,} documents.")
    return conn # Return the live SQLite connection


def build_index(dataset_dir: str, db_path: str, local_db: str, force_rebuild: bool = False):

  # Connect to the local SQLite DB (creates if not exists)
    conn = sqlite3.connect(local_db, timeout=30)
    c    = conn.cursor()

    # Check if 'posts' table already exists
    c.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='posts'")
    already_exists = c.fetchone() is not None
    if already_exists and not force_rebuild:

      # If index exists and rebuild not forced, skip building
        c.execute("SELECT COUNT(*) FROM posts")
        count = c.fetchone()[0]
        print(f"Index already exists with {count:,} documents. Skipping rebuild.")
        return conn
    print("Building index from scratch...")
    c.execute("DROP TABLE IF EXISTS posts")
    c.execute("DROP TABLE IF EXISTS posts_lookup")
    # Create FTS5 virtual table for full-text search (porter stemming)
    c.execute("""
        CREATE VIRTUAL TABLE posts USING fts5(
            doc_id   UNINDEXED,
            content,
            tokenize = 'porter'
        )
    """)
    # Lookup table for fast doc_id → content retrieval
    c.execute("""
        CREATE TABLE IF NOT EXISTS posts_lookup (
            doc_id  TEXT PRIMARY KEY,
            content TEXT
        )
    """)
    conn.commit()

    # Batch setup for efficient inserts
    BATCH_SIZE   = 10_000
    batch        = []
    lookup_batch = []
    total_docs   = 0
    total_skip   = 0

    # Sort .trec files numerically to process in order
    trec_files_sorted = sorted(
        [f for f in os.listdir(dataset_dir) if f.endswith(".trec")],
        key=lambda f: int(re.search(r"\d+", f).group())
    )

    # Process each file and document
    for filename in tqdm(trec_files_sorted, desc="Indexing files"):
        filepath = os.path.join(dataset_dir, filename)
        for doc in parse_trec_file_stream(filepath):
          # Combine pre, text, post sections and clean text
            combined = " ".join(filter(None, [doc.get("pre",""), doc.get("text",""), doc.get("post","")]))
            cleaned  = preprocess(combined)
            if not cleaned:
                total_skip += 1
                continue

            # Add to batch for FTS table and lookup table
            batch.append((doc["doc_id"], cleaned))
            lookup_batch.append((doc["doc_id"], cleaned))
            total_docs += 1

            # Insert batches to improve performance
            if len(batch) >= BATCH_SIZE:
                c.executemany("INSERT INTO posts(doc_id, content) VALUES (?,?)", batch)
                c.executemany("INSERT OR IGNORE INTO posts_lookup VALUES (?,?)", lookup_batch)
                conn.commit()
                batch        = []
                lookup_batch = []

    # Insert remaining documents
    if batch:
        c.executemany("INSERT INTO posts(doc_id, content) VALUES (?,?)", batch)
        c.executemany("INSERT OR IGNORE INTO posts_lookup VALUES (?,?)", lookup_batch)
        conn.commit()

     # Print summary of indexing
    print(f"\nIndexing complete! Documents: {total_docs:,} | Skipped: {total_skip:,}")
    shutil.copy(local_db, db_path)
    print(f"Saved to {db_path}")
    return conn

# Build or load the index
FORCE_REBUILD = False
conn = build_index(DATASET_DIR, DB_PATH, LOCAL_DB, force_rebuild=FORCE_REBUILD)
c    = conn.cursor()

Index already exists with 17,512,342 documents. Skipping rebuild.


## BDI Queries + BM25 Baseline



> BM25 is a classic probabilistic retrieval model that ranks documents by term matching on keyword overlap.

**Role in the pipeline**
* Serves as the initial retrieval stage over the full corpus.
* Retrieves the top‑k (e.g., 1,000) sentence candidates per symptom query.
* Creates a baseline run that captures lexical relevance.


This code performs a BM25 retrieval baseline on the preprocessed eRisk posts using Beck Depression Inventory (BDI) symptom-based queries.

**BDI queries and symptoms:**
* BDI_QUERIES contains keyword-based queries for each of the 21 BDI items.
* BDI_SYMPTOMS provides a human-readable description for each BDI item.

**Query formatting:**
* format_fts5_query() preprocesses each BDI query and converts it into a
* FTS5-compatible OR query for full-text search.

**Run BM25 retrieval:**
* run_bm25_queries() executes each BDI query against the posts FTS5 table using BM25 scoring.
* Retrieves the top-k documents (default 1000) per query.
* Writes results in standard TREC format: query_id Q0 doc_id rank score run_name.
* Collects statistics on the number of results per query.

In [ ]:
BDI_QUERIES = {
    "1":  "sad unhappy miserable sorrow depressed down low spirits gloomy",
    "2":  "hopeless future pointless nothing better doomed giving up despair",
    "3":  "failure failed regret disappointed mistake loser inadequate",
    "4":  "nothing fun anymore lost interest used to love now empty numb hollow",
    "5":  "guilt guilty ashamed remorse blame horrible person",
    "6":  "deserve punishment deserve pain bad person being punished",
    "7":  "hate myself loathe despise self hate disgusted with myself",
    "8":  "self criticism self blame harsh judge myself too hard on myself",
    "9":  "suicidal thoughts suicide death wish end life kill myself",
    "10": "crying cry tears sob sobbing weeping breakdown",
    "11": "restless nervous pacing cannot relax sit still anxious tense",
    "12": "withdrawn isolated stopped caring gave up dont care anymore",
    "13": "cannot decide confused uncertain second guess overthinking",
    "14": "worthless useless burden nobody cares unwanted dead weight",
    "15": "drained low energy sluggish lethargic no motivation heavy",
    "16": "insomnia cannot sleep waking early oversleeping tired all time",
    "17": "irritable angry annoyed frustrated short temper snapping",
    "18": "lost appetite not eating overeating food no hunger",
    "19": "cannot focus brain fog distracted cannot think concentrate",
    "20": "exhausted tired fatigue worn out physically drained",
    "21": "lost interest intimacy no desire physical connection",
}

BDI_SYMPTOMS = {
    "1":  "Sadness — feeling sad, empty, or hopeless",
    "2":  "Pessimism — feeling discouraged about the future",
    "3":  "Past Failure — feeling like a failure",
    "4":  "Loss of Pleasure — not getting pleasure from things",
    "5":  "Guilty Feelings — feeling guilty",
    "6":  "Punishment Feelings — feeling like being punished",
    "7":  "Self-Dislike — feeling disappointed in oneself",
    "8":  "Self-Criticalness — blaming oneself for things",
    "9":  "Suicidal Thoughts — thoughts of killing oneself",
    "10": "Crying — crying more than usual",
    "11": "Agitation — feeling restless or agitated",
    "12": "Loss of Interest — losing interest in things",
    "13": "Indecisiveness — difficulty making decisions",
    "14": "Worthlessness — feeling worthless",
    "15": "Loss of Energy — not having enough energy",
    "16": "Sleep Changes — sleeping too much or too little",
    "17": "Irritability — feeling irritable or angry",
    "18": "Appetite Changes — eating too much or too little",
    "19": "Concentration Difficulty — trouble concentrating",
    "20": "Tiredness — feeling tired all the time",
    "21": "Loss of Interest in Sex — less interested in sex",
}

# Step 1: Format a query for SQLite FTS5
def format_fts5_query(query: str) -> str:
    cleaned = preprocess(query)
    tokens  = cleaned.strip().split()
    return " OR ".join(f'"{t}"' for t in tokens)

# Step 2: Run BM25 queries against SQLite FTS5 table
def run_bm25_queries(cursor, queries, output_path, run_name, top_k=1000):
    stats = {}
    with open(output_path, "w") as f_out:
        for qid, raw_query in queries.items():
            fts5_query = format_fts5_query(raw_query)
            try:
              # Retrieve top-k documents ranked by BM25
                cursor.execute("""
                    SELECT doc_id, -bm25(posts) AS score
                    FROM   posts
                    WHERE  posts MATCH ?
                    ORDER  BY score DESC
                    LIMIT  ?
                """, (fts5_query, top_k))
                results    = cursor.fetchall()
                stats[qid] = len(results)

                # Write TREC-style run file
                for rank, (doc_id, score) in enumerate(results, start=1):
                    f_out.write(f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_name}\n")
            except Exception as e:
                print(f"Query {qid} failed: {e}")
                stats[qid] = 0
    return stats

# Step 3: Execute BM25 baseline and report
print("Running BM25 baseline queries...")
bm25_stats = run_bm25_queries(c, BDI_QUERIES, RUN_BM25, "BM25_Baseline", top_k=1000)

print(f"\n=== BM25 run complete ===")
for qid, count in bm25_stats.items():
    flag = "" if count < 100 else "  "
    print(f"  {flag} Query {qid:>2} : {count:,} results")

Running BM25 baseline queries...

=== BM25 run complete ===
     Query  1 : 1,000 results
     Query  2 : 1,000 results
     Query  3 : 1,000 results
     Query  4 : 1,000 results
     Query  5 : 1,000 results
     Query  6 : 1,000 results
     Query  7 : 1,000 results
     Query  8 : 1,000 results
     Query  9 : 1,000 results
     Query 10 : 1,000 results
     Query 11 : 1,000 results
     Query 12 : 1,000 results
     Query 13 : 1,000 results
     Query 14 : 1,000 results
     Query 15 : 1,000 results
     Query 16 : 1,000 results
     Query 17 : 1,000 results
     Query 18 : 1,000 results
     Query 19 : 1,000 results
     Query 20 : 1,000 results
     Query 21 : 1,000 results


## Convert qrels to TREC Format + Evaluate BM25

* **Write TREC qrels:** Saves qrels_rows to a TREC-style relevance file (query_id 0 doc_id relevance).
* **Setup trec_eval:** Clones and compiles the TREC evaluation tool if it’s not already available.
* **Evaluate BM25:** Runs trec_eval to compute MAP, nDCG@10, and Precision@10 for the BM25 retrieval results.

In [ ]:
# Clone the official TREC evaluation repo
with open(QRELS_TREC, "w") as f:
    for row in qrels_rows:
        f.write(f"{row['query']} 0 {row['doc_id']} {int(row['relevant'])}\n")
print(f"TREC qrels written — {len(qrels_rows):,} lines")

if not os.path.exists("trec_eval"):
    !git clone https://github.com/usnistgov/trec_eval.git
    # Compile trec_eval
    !make -C trec_eval

# Step 3: Evaluate BM25 run using trec_eval
print("\n" + "="*50)
print("  BM25 Baseline")
print("="*50)
!./trec_eval/trec_eval -J -m map -m ndcg_cut.10 -m P.10 {QRELS_TREC} {RUN_BM25}

TREC qrels written — 11,042 lines

  BM25 Baseline
map                   	all	0.0515
P_10                  	all	0.5905
ndcg_cut_10           	all	0.6240


## Install SBERT + FAISS

In [ ]:
!pip install sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer, util
import faiss
import torch

print(f"GPU available: {torch.cuda.is_available()}")

GPU available: True


## BM25 + SBERT Retrieve and Rerank



> Sentence Transformers (e.g., MiniLM) encode queries and candidate sentences into dense vectors so that semantically similar text has higher cosine similarity.


The code performs BM25 retrieval followed by semantic reranking using a sentence embedding model:

* Load the model – a SentenceTransformer like MiniLM.
* Retrieve candidates – run BM25 on an SQLite FTS5 database to get top documents for each query.
* Compute embeddings – encode both the query and candidate documents into vector representations.
* Rerank by similarity – calculate cosine similarity between the query and document embeddings and rank the top candidates.
* Save results – write the reranked documents to a TREC-style output file, keeping track of how many documents were reranked per query.

In [ ]:
# Step: Retrieve top-k BM25 candidates and rerank with embeddings
MODELS = {"minilm": "all-MiniLM-L6-v2"}

def retrieve_and_rerank(cursor, queries, model_name, run_name, output_path,
                         bm25_top_k=1000, rerank_top_k=1000):
  """
    Performs a two-stage retrieval:
      1. BM25 retrieval from SQLite FTS5 table (top `bm25_top_k` docs)
      2. Reranking with a SentenceTransformer embedding model (top `rerank_top_k` docs)

    Args:
        cursor      : SQLite cursor
        queries     : dict of {query_id: query_text}
        model_name  : embedding model name
        run_name    : label for TREC run file
        output_path : path to write TREC-style run file
        bm25_top_k  : number of BM25 candidates to retrieve
        rerank_top_k: number of top candidates to write after reranking

    Returns:
        stats: dict of query_id -> number of reranked docs
    """
    print(f"\nLoading model: {model_name}")
    model = SentenceTransformer(model_name)
    stats = {}
    with open(output_path, "w") as f_out:
        for qid, raw_query in queries.items():
            print(f"  Processing Query {qid}...", end=" ")
            fts5_query = format_fts5_query(raw_query)

            # Retrieve BM25 candidates
            try:
                cursor.execute("""
                    SELECT doc_id, content, -bm25(posts) AS bm25_score
                    FROM   posts WHERE posts MATCH ?
                    ORDER  BY bm25_score DESC LIMIT ?
                """, (fts5_query, bm25_top_k))
                candidates = cursor.fetchall()
            except Exception as e:
                print(f"BM25 failed: {e}")
                stats[qid] = 0
                continue
            if not candidates:
                print("No candidates!")
                stats[qid] = 0
                continue

            # Prepare for embedding-based reranking
            doc_ids = [r[0] for r in candidates]
            texts   = [r[1] for r in candidates]

            # Encode query and documents
            query_vec = model.encode(raw_query, convert_to_tensor=True, normalize_embeddings=True)
            doc_vecs  = model.encode(texts, batch_size=128, convert_to_tensor=True,
                                     normalize_embeddings=True, show_progress_bar=False)

            # Compute cosine similarity scores
            scores         = util.cos_sim(query_vec, doc_vecs)[0]
            ranked_indices = torch.argsort(scores, descending=True)

            # Write top-k reranked documents in TREC format
            count = 0
            for rank, idx in enumerate(ranked_indices[:rerank_top_k], start=1):
                f_out.write(f"{qid} Q0 {doc_ids[idx.item()]} {rank} {scores[idx.item()].item():.6f} {run_name}\n")
                count += 1
            stats[qid] = count
            print(f"{len(candidates):,} → reranked {count:,}")
    print(f"\n{run_name} complete! → {output_path}")
    return stats

#Execute reranking for all specified models
rerank_stats = {}
for model_key, (output_path, run_name, model_name) in {
    "minilm": (RUN_RERANK, "Rerank_MiniLM", MODELS["minilm"])
}.items():
    stats = retrieve_and_rerank(c, BDI_QUERIES, model_name, run_name, output_path,
                                 bm25_top_k=1000, rerank_top_k=1000)
    rerank_stats[model_key] = stats


Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Processing Query 1... 1,000 → reranked 1,000
  Processing Query 2... 1,000 → reranked 1,000
  Processing Query 3... 1,000 → reranked 1,000
  Processing Query 4... 1,000 → reranked 1,000
  Processing Query 5... 1,000 → reranked 1,000
  Processing Query 6... 1,000 → reranked 1,000
  Processing Query 7... 1,000 → reranked 1,000
  Processing Query 8... 1,000 → reranked 1,000
  Processing Query 9... 1,000 → reranked 1,000
  Processing Query 10... 1,000 → reranked 1,000
  Processing Query 11... 1,000 → reranked 1,000
  Processing Query 12... 1,000 → reranked 1,000
  Processing Query 13... 1,000 → reranked 1,000
  Processing Query 14... 1,000 → reranked 1,000
  Processing Query 15... 1,000 → reranked 1,000
  Processing Query 16... 1,000 → reranked 1,000
  Processing Query 17... 1,000 → reranked 1,000
  Processing Query 18... 1,000 → reranked 1,000
  Processing Query 19... 1,000 → reranked 1,000
  Processing Query 20... 1,000 → reranked 1,000
  Processing Query 21... 1,000 → reranked 1,000



## Hybrid Search (BM25 + SBERT)

> A hybrid approach combines both BM25 lexical scores and semantic embedding scores to leverage both

The code implements a hybrid retrieval system combining BM25 and semantic embeddings:

* Load the model – a SentenceTransformer like MiniLM for semantic similarity.
* Retrieve BM25 candidates – query an SQLite FTS5 database to get the top documents per query.
* Normalize BM25 scores – scale BM25 scores between 0 and 1 to make them compatible with embedding scores.
* Compute embeddings – encode the query and candidate documents into vector representations.
* Combine scores – calculate cosine similarity scores from embeddings and mix them with normalized BM25 scores using specified weights (bm25_weight and sbert_weight).
* Rank documents – sort candidates by the hybrid score and select the top top_k.

In [ ]:
def hybrid_search(cursor, queries, model_name, run_name, output_path,
                  bm25_weight=0.2, sbert_weight=0.8, top_k=1000):
  """
    Performs hybrid ranking:
      1. Retrieve top-k BM25 candidates from SQLite FTS5 table
      2. Compute embeddings via SentenceTransformer and get cosine similarity
      3. Combine BM25 and SBERT scores into a single hybrid score
      4. Write top-k results in TREC format

    Args:
        cursor      : SQLite cursor
        queries     : dict of {query_id: query_text}
        model_name  : embedding model name
        run_name    : run label for TREC file
        output_path : path to write TREC-style run file
        bm25_weight : weight for BM25 score
        sbert_weight: weight for embedding similarity
        top_k       : maximum number of docs per query

    Returns:
        stats: dict of query_id -> number of scored docs
    """
    print(f"\nLoading model: {model_name}")
    model = SentenceTransformer(model_name)
    stats = {}
    with open(output_path, "w") as f_out:
        for qid, raw_query in queries.items():
            print(f"  Processing Query {qid}...", end=" ")

            # Format query for FTS5 BM25
            fts5_query = format_fts5_query(raw_query)

            # Retrieve BM25 candidates
            try:
                cursor.execute("""
                    SELECT doc_id, content, -bm25(posts) AS bm25_score
                    FROM   posts WHERE posts MATCH ?
                    ORDER  BY bm25_score DESC LIMIT ?
                """, (fts5_query, top_k))
                candidates = cursor.fetchall()
            except Exception as e:
                print(f"BM25 failed: {e}")
                stats[qid] = 0
                continue
            if not candidates:
                stats[qid] = 0
                continue

            # Extract doc info and normalize BM25 scores
            doc_ids     = [r[0] for r in candidates]
            texts       = [r[1] for r in candidates]
            bm25_scores = np.array([r[2] for r in candidates])
            bm25_min    = bm25_scores.min()
            bm25_max    = bm25_scores.max()
            bm25_norm   = (bm25_scores - bm25_min) / (bm25_max - bm25_min) \
                          if bm25_max - bm25_min > 0 else np.ones_like(bm25_scores)

            # Compute embeddings and cosine similarity scores
            query_vec     = model.encode(raw_query, convert_to_tensor=True, normalize_embeddings=True)
            doc_vecs      = model.encode(texts, batch_size=128, convert_to_tensor=True,
                                         normalize_embeddings=True, show_progress_bar=False)
            sbert_scores  = util.cos_sim(query_vec, doc_vecs)[0].cpu().numpy()

            # Compute hybrid score = weighted BM25 + SBERT
            hybrid_scores = (bm25_weight * bm25_norm) + (sbert_weight * sbert_scores)

            # Rank documents by hybrid score
            ranked_idx    = np.argsort(hybrid_scores)[::-1]
            for rank, idx in enumerate(ranked_idx[:top_k], start=1):
                f_out.write(f"{qid} Q0 {doc_ids[idx]} {rank} {hybrid_scores[idx]:.6f} {run_name}\n")
            stats[qid] = min(len(candidates), top_k)
            print(f"{len(candidates):,} → hybrid scored")
    print(f"\n{run_name} complete! → {output_path}")
    return stats

#Run hybrid BM25 + MiniLM scoring
hybrid_stats = hybrid_search(c, BDI_QUERIES, MODELS["minilm"], "Hybrid_BM25_MiniLM",
                              RUN_HYBRID, bm25_weight=0.2, sbert_weight=0.8, top_k=1000)


Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Processing Query 1... 1,000 → hybrid scored
  Processing Query 2... 1,000 → hybrid scored
  Processing Query 3... 1,000 → hybrid scored
  Processing Query 4... 1,000 → hybrid scored
  Processing Query 5... 1,000 → hybrid scored
  Processing Query 6... 1,000 → hybrid scored
  Processing Query 7... 1,000 → hybrid scored
  Processing Query 8... 1,000 → hybrid scored
  Processing Query 9... 1,000 → hybrid scored
  Processing Query 10... 1,000 → hybrid scored
  Processing Query 11... 1,000 → hybrid scored
  Processing Query 12... 1,000 → hybrid scored
  Processing Query 13... 1,000 → hybrid scored
  Processing Query 14... 1,000 → hybrid scored
  Processing Query 15... 1,000 → hybrid scored
  Processing Query 16... 1,000 → hybrid scored
  Processing Query 17... 1,000 → hybrid scored
  Processing Query 18... 1,000 → hybrid scored
  Processing Query 19... 1,000 → hybrid scored
  Processing Query 20... 1,000 → hybrid scored
  Processing Query 21... 1,000 → hybrid scored

Hybrid_BM25_MiniLM co

## Load Mistral-7B with 4-bit Quantization


> The final stage uses a large language model (Mistral‑7B) to score relevance via prompt‑based inference



The code loads Mistral-7B-Instruct in 4-bit quantized mode for efficient inference:

**Setup 4-bit quantization:**
* Uses BitsAndBytesConfig to load the model in 4-bit precision with double quantization (bnb_4bit_use_double_quant) and compute in float16.
* Reduces memory usage while keeping reasonable accuracy.

**Load tokenizer:**
* Uses AutoTokenizer for the Mistral model.
* Sets pad_token to the end-of-sequence token for compatibility.

**Load model with quantization:**
* Uses AutoModelForCausalLM with `quantization_config=bnb_config`.
* `device_map="auto"` automatically places layers on available GPUs


**Set evaluation mode:**
* `mistral_model.eval() `disables training behaviors like dropout.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit             = True,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype   = torch.float16,
    bnb_4bit_quant_type      = "nf4"
)

mistral_model_card = "mistralai/Mistral-7B-Instruct-v0.3"

#Load tokenizer
mistral_tokenizer = AutoTokenizer.from_pretrained(
    mistral_model_card,
    use_fast=True
)

# Set padding token to EOS to avoid errors during batch processing
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token

#Load 4-bit quantized Mistral model
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_card,
    quantization_config = bnb_config,
    device_map          = "auto"
)

mistral_model.eval()
print("Mistral-7B loaded with 4-bit quantization!")
print(f"Model device: {next(mistral_model.parameters()).device}")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral-7B loaded with 4-bit quantization!
Model device: cuda:0


## Hardcoded Few-Shot Examples

The code defines few-shot examples for 21 BDI (Beck Depression Inventory) items to support classification or retrieval tasks:

**Structure:**
* Each key ("1" to "21") represents a BDI item (e.g., Sadness, Pessimism, Loss of Energy).

**Each item contains:**
* relevant → examples reflecting the symptom.
* not_relevant → examples not indicative of the symptom.

**Purpose:**
* Provides few-shot guidance for models to distinguish between relevant and irrelevant expressions of each BDI item.

In [ ]:
FEW_SHOT_EXAMPLES = {
    "1": {  # Sadness
        "relevant": [
            "I've been feeling so empty inside lately. Like there's this grey cloud that just follows me everywhere and I can't shake it.",
            "Woke up crying again for no reason. Just this deep sadness I can't explain to anyone around me."
        ],
        "not_relevant": [
            "That movie was so sad, cried the whole way through. 10/10 would recommend it though.",
            "My dog seemed really sad when I left for work this morning, poor thing."
        ]
    },
    "2": {  # Pessimism
        "relevant": [
            "I don't see the point of trying anymore. No matter what I do things just get worse. There is no future for me.",
            "Everyone keeps telling me it will get better but I genuinely don't believe that. Nothing has ever gotten better for me."
        ],
        "not_relevant": [
            "I'm a bit pessimistic about the economy this year, inflation just keeps getting worse.",
            "My dad is always pessimistic about everything, it's exhausting to be around him sometimes."
        ]
    },
    "3": {  # Past Failure
        "relevant": [
            "I keep thinking about all the times I have failed. Failed at school, failed at relationships, failed at my career. I'm just a failure.",
            "Can't stop replaying every mistake I've ever made. Every bad decision just loops in my head over and over."
        ],
        "not_relevant": [
            "The company failed to meet its quarterly targets which was disappointing for investors.",
            "My friend failed her driving test again, third time now. She was pretty upset about it."
        ]
    },
    "4": {  # Loss of Pleasure
        "relevant": [
            "I used to love going out with friends, watching movies, playing guitar. Now none of it brings me any joy. It's all just empty.",
            "Gaming was my escape for years. Now I sit down to play and I feel absolutely nothing. Just going through the motions."
        ],
        "not_relevant": [
            "I lost my wallet last week but found it yesterday, such a relief.",
            "Our team lost the match but we still had a lot of fun playing together."
        ]
    },
    "5": {  # Guilty Feelings
        "relevant": [
            "I feel so guilty all the time. Like I've done something wrong even when I haven't. This guilt is eating me alive.",
            "I can't stop feeling guilty about things that happened years ago. I know I should forgive myself but I just can't."
        ],
        "not_relevant": [
            "I feel a bit guilty about forgetting my friend's birthday, going to send her a gift to make up for it.",
            "He felt guilty about lying to his parents but eventually came clean and told them the truth."
        ]
    },
    "6": {  # Punishment Feelings
        "relevant": [
            "I genuinely feel like everything bad that happens to me is what I deserve. Like the universe is punishing me for being a bad person.",
            "I don't try to avoid pain anymore. Part of me feels like I deserve to suffer and there's no point fighting it."
        ],
        "not_relevant": [
            "The employee received a formal punishment for repeatedly breaking company policy.",
            "I think the punishment for that particular crime was way too lenient honestly."
        ]
    },
    "7": {  # Self-Dislike
        "relevant": [
            "I look in the mirror and I hate what I see. Not just physically but who I am as a person. I genuinely disgust myself.",
            "I hate myself. Not in a casual way. I genuinely despise who I am and I don't know how to stop feeling this way."
        ],
        "not_relevant": [
            "I hate Monday mornings, can never wake up on time no matter what I do.",
            "She hated herself for eating the whole cake but laughed it off with her friends afterwards."
        ]
    },
    "8": {  # Self-Criticalness
        "relevant": [
            "I am so hard on myself. Every tiny mistake I make I replay it for days, calling myself stupid and completely useless.",
            "My internal voice is just constant criticism. I can't do anything right according to my own brain. It never stops."
        ],
        "not_relevant": [
            "My boss is really self-critical, always second-guessing his own decisions at work.",
            "Athletes need to be critical of their own performance if they want to keep improving."
        ]
    },
    "9": {  # Suicidal Thoughts
        "relevant": [
            "I keep having thoughts about not being here anymore. Like the world would be better off without me. I'm scared of these thoughts.",
            "Sometimes I think about ending it all. I don't know what to do with these thoughts and I'm completely exhausted."
        ],
        "not_relevant": [
            "The character in the book contemplated suicide which made it a powerful but very difficult read.",
            "We need much better mental health education and suicide prevention programs in schools."
        ]
    },
    "10": {  # Crying
        "relevant": [
            "I cry every single day now. Sometimes for hours. I can't control it and I don't even always know why it starts.",
            "Burst into tears at work today over absolutely nothing. Had to hide in the bathroom. I feel like I'm falling apart."
        ],
        "not_relevant": [
            "That wedding speech made literally everyone cry, it was so beautifully written.",
            "My little sister cried the whole flight, she's only two so it was completely expected."
        ]
    },
    "11": {  # Agitation
        "relevant": [
            "I can't sit still. I pace around my apartment for hours. There's this restless energy inside me that I just can't get rid of.",
            "I feel so agitated all the time lately. Like my skin is crawling and I cannot calm down no matter what I try."
        ],
        "not_relevant": [
            "The crowd got really agitated when the concert was delayed by two hours with no explanation.",
            "He gets agitated in traffic but is generally calm and relaxed when he's at home."
        ]
    },
    "12": {  # Loss of Interest
        "relevant": [
            "I've lost interest in everything. My hobbies, my friends, my job. I just don't care about any of it anymore.",
            "Nothing interests me anymore. I used to have so many things I was passionate about. Now it's all completely gone."
        ],
        "not_relevant": [
            "I've lost interest in that TV show, the new season just isn't as good as the first.",
            "The investors lost interest in the project after seeing the revised budget estimates."
        ]
    },
    "13": {  # Indecisiveness
        "relevant": [
            "I can't make any decisions anymore. Even small things like what to eat for dinner become completely paralysing for me.",
            "My brain just won't commit to anything. I go back and forth for hours on the simplest choices. It's absolutely exhausting."
        ],
        "not_relevant": [
            "The committee was indecisive about the new policy and kept postponing the vote indefinitely.",
            "She's always been a bit indecisive about fashion, tries on ten outfits before picking one."
        ]
    },
    "14": {  # Worthlessness
        "relevant": [
            "I am worthless. I contribute nothing to anyone's life and I know it. Nobody would miss me if I was gone.",
            "I feel like such a burden to everyone around me. Completely useless and worthless. I don't know why anyone puts up with me."
        ],
        "not_relevant": [
            "That old car is completely worthless now, not worth the cost of repairing it.",
            "The counterfeit notes were worthless and police were warning the public to be vigilant."
        ]
    },
    "15": {  # Loss of Energy
        "relevant": [
            "I have absolutely zero energy. Getting out of bed takes everything I have. I'm exhausted before the day even starts.",
            "I've been so drained lately. Even the simplest tasks feel like climbing a mountain. I don't know what's wrong with me."
        ],
        "not_relevant": [
            "The power outage left the whole neighbourhood without energy for about six hours.",
            "My phone battery loses energy so quickly now, I definitely need to get a new one."
        ]
    },
    "16": {  # Sleep Changes
        "relevant": [
            "I can't sleep at all. I lie in bed for hours, my mind just won't shut off. Then when I do finally sleep I wake up at 3am.",
            "I've been sleeping 12 to 14 hours a day and I'm still exhausted all the time. I just can't face being awake."
        ],
        "not_relevant": [
            "The new baby has completely changed our sleep schedule, new parent life is tough.",
            "Researchers found that sleep patterns change quite significantly with age according to the study."
        ]
    },
    "17": {  # Irritability
        "relevant": [
            "I snap at everyone lately. My family, my friends. The smallest things set me off and then I feel terrible about it afterwards.",
            "I've been so irritable all the time. Everything annoys me. I hate feeling like this, like I'm always ready to explode."
        ],
        "not_relevant": [
            "The new software update is so irritating, they moved everything around for no reason.",
            "Traffic always makes people irritable especially on hot days, it's pretty normal."
        ]
    },
    "18": {  # Appetite Changes
        "relevant": [
            "I haven't eaten a proper meal in days. I'm not even hungry, food just doesn't appeal to me at all anymore.",
            "I've been stress eating so much lately. Stuffing myself even when I'm full. I don't know how to stop doing this."
        ],
        "not_relevant": [
            "This restaurant has huge portions, my appetite wasn't nearly enough for even half of it.",
            "Dogs apparently have bigger appetites in colder weather, I didn't know that before."
        ]
    },
    "19": {  # Concentration Difficulty
        "relevant": [
            "I can't focus on anything. I'll read the same paragraph five times and none of it goes in. My brain is completely foggy.",
            "Concentration is basically impossible for me right now. I sit down to work and my mind just wanders away endlessly."
        ],
        "not_relevant": [
            "The students had trouble concentrating during the exam because of construction noise outside.",
            "My cat has impressive concentration when watching birds through the window, it's quite funny."
        ]
    },
    "20": {  # Tiredness
        "relevant": [
            "I am so tired all the time. Not just sleepy tired, but bone-deep exhausted. No amount of sleep makes any difference.",
            "I wake up tired and I go to bed tired. I'm tired of being tired. I don't even remember what having energy felt like."
        ],
        "not_relevant": [
            "I was so tired after the marathon yesterday, slept for a solid ten hours straight.",
            "She sounded tired on the phone, but it was probably just a long day at work."
        ]
    },
    "21": {  # Loss of Interest in Sex
        "relevant": [
            "My partner and I barely connect physically anymore. I've completely lost any desire for intimacy and it's been months now.",
            "I have zero interest in sex or any kind of physical closeness lately. I feel terrible for my partner but I just can't."
        ],
        "not_relevant": [
            "Studies show that interest in sex naturally decreases with age in long-term relationships apparently.",
            "The two characters in the novel had completely different attitudes toward intimacy which caused a lot of tension."
        ]
    },
}

print(f"FEW_SHOT_EXAMPLES loaded for {len(FEW_SHOT_EXAMPLES)} BDI items.")
print("\nSample — Query 1 (Sadness):")
for label, examples in FEW_SHOT_EXAMPLES["1"].items():
    for ex in examples:
        print(f"  [{label}] {ex[:80]}...")

FEW_SHOT_EXAMPLES loaded for 21 BDI items.

Sample — Query 1 (Sadness):
  [relevant] I've been feeling so empty inside lately. Like there's this grey cloud that just...
  [relevant] Woke up crying again for no reason. Just this deep sadness I can't explain to an...
  [not_relevant] That movie was so sad, cried the whole way through. 10/10 would recommend it tho...
  [not_relevant] My dog seemed really sad when I left for work this morning, poor thing....


## Few-Shot Prompt Template

In [ ]:
few_shot_prompt = [
    {
        "role": "system",
        "content": "You are an expert annotator for detecting symptoms of depression in social media text."
    },
    {
        "role": "user",
        "content": """Your task is to determine whether the input sentence is relevant to a specific depression symptom from the BDI (Beck Depression Inventory) questionnaire.

A sentence is RELEVANT if it provides information about the USER'S OWN condition regarding the symptom — including when the user says they are okay or do not have the symptom.
A sentence is NOT RELEVANT if it is about someone else, fictional, hypothetical, or completely unrelated to the symptom.

SYMPTOM: {symptom}

EXAMPLES:
{examples}

Respond only by writing one of the following labels:
relevant, not-relevant

TEXT: {text}

ANSWER"""
    }
]

print("Few-shot prompt template defined.")

Few-shot prompt template defined.


## prepare_few_shot_prompts

> Uses hardcoded `FEW_SHOT_EXAMPLES` — no dependency on qrels.

In [ ]:
import random

def prepare_few_shot_prompts(texts, symptom_desc, prompt_template, tokenizer, qid,
                              shuffle_examples=True, seed=42):
    """
    Format texts into few-shot Mistral instruction prompts.

    Uses hardcoded FEW_SHOT_EXAMPLES (no qrels) so evaluation stays clean.
    The same demonstrations are used for all texts within a query — consistent
    and reproducible.

    Inputs:
        texts          : list of candidate post texts to score
        symptom_desc   : BDI symptom description (from BDI_SYMPTOMS)
        prompt_template: the few_shot_prompt template
        tokenizer      : Mistral tokenizer
        qid            : BDI query ID (e.g. '1') — selects the right examples
        shuffle_examples: shuffle the 4 demo examples (keeps model honest)
        seed           : random seed for reproducibility

    Outputs:
        list of formatted instruction strings, one per text
    """
    # Build the demonstrations string once for the whole query
    examples = FEW_SHOT_EXAMPLES[qid]
    demo_parts = []
    for text in examples["relevant"]:
        demo_parts.append(f"TEXT: {text}\nANSWER: relevant")
    for text in examples["not_relevant"]:
        demo_parts.append(f"TEXT: {text}\nANSWER: not-relevant")

    if shuffle_examples:
        random.seed(seed)
        random.shuffle(demo_parts)

    demonstrations_string = "\n\n".join(demo_parts)
    user_content_base     = prompt_template[1]["content"]

    prompts = []
    for text in texts:
        user_content = (
            user_content_base
            .replace("{symptom}",  symptom_desc)
            .replace("{examples}", demonstrations_string)
            .replace("{text}",     text[:400])
        )
        prompt = [
            prompt_template[0],
            {"role": "user", "content": user_content}
        ]
        try:
            formatted = tokenizer.apply_chat_template(
                prompt, tokenize=False, add_generation_prompt=True
            )
            prompts.append(formatted)
        except Exception as e:
            print(f"Warning: prompt formatting failed — {e}")
            prompts.append(None)

    return prompts


# ── Quick test ──────────────────────────────────────────────────────────────
test_prompts = prepare_few_shot_prompts(
    texts           = ["I feel so empty and hollow inside, nothing brings me joy anymore."],
    symptom_desc    = BDI_SYMPTOMS["1"],
    prompt_template = few_shot_prompt,
    tokenizer       = mistral_tokenizer,
    qid             = "1",
)
print(f"Prompts generated: {len(test_prompts)}")
print(f"\nPreview (first 700 chars):")
print(test_prompts[0][:700])

Prompts generated: 1

Preview (first 700 chars):
<s>[INST] You are an expert annotator for detecting symptoms of depression in social media text.

Your task is to determine whether the input sentence is relevant to a specific depression symptom from the BDI (Beck Depression Inventory) questionnaire.

A sentence is RELEVANT if it provides information about the USER'S OWN condition regarding the symptom — including when the user says they are okay or do not have the symptom.
A sentence is NOT RELEVANT if it is about someone else, fictional, hypothetical, or completely unrelated to the symptom.

SYMPTOM: Sadness — feeling sad, empty, or hopeless

EXAMPLES:
TEXT: That movie was so sad, cried the whole way through. 10/10 would recommend it thou


## Token ID Setup + `score_relevance_logits`

**Token setup:**
* Encodes the words " relevant" and " not" to obtain their token IDs in the tokenizer.
* Ensures they are single tokens and warns if tokenization splits them into multiple.
* These tokens are used as the classification targets.

**Score computation (score_relevance_logits):**
* Inputs: the model, tokenizer, a list of prompts, and batch size.

**For each batch of prompts:**
* Tokenize and move to the model device.
* Forward pass through the model (no generation).
* Extract logits of the last real token in each prompt.
* Compute log-probabilities for the "relevant" and "not" tokens.
* Apply a 2-class softmax to get a probability that the prompt indicates "relevant".
* Returns a list of float scores in [0,1].

**Sanity check:**
* Prepares a few-shot prompt for BDI item 1 (Sadness) using _test_texts.
* Scores each text for relevance to Sadness.
* Prints the scores: high values indicate likely relevant, low values indicate not relevant.

In [ ]:
_rel_tokens = mistral_tokenizer.encode(" relevant", add_special_tokens=False)
_not_tokens = mistral_tokenizer.encode(" not",      add_special_tokens=False)

RELEVANT_TOKEN_ID = _rel_tokens[0]
NOT_TOKEN_ID      = _not_tokens[0]

print(f"Token for ' relevant'  : id={RELEVANT_TOKEN_ID:6d}  decoded={mistral_tokenizer.decode([RELEVANT_TOKEN_ID])!r}")
print(f"Token for ' not'       : id={NOT_TOKEN_ID:6d}  decoded={mistral_tokenizer.decode([NOT_TOKEN_ID])!r}")
print()

# Sanity check — making sure these are single tokens
assert len(_rel_tokens) >= 1, "' relevant' tokenised to zero tokens — unexpected"
assert len(_not_tokens) >= 1, "' not' tokenised to zero tokens — unexpected"
if len(_rel_tokens) > 1:
    print(f"Note: ' relevant' tokenised to {len(_rel_tokens)} tokens; using only first token id={RELEVANT_TOKEN_ID}")
if len(_not_tokens) > 1:
    print(f"Note: ' not' tokenised to {len(_not_tokens)} tokens; using only first token id={NOT_TOKEN_ID}")


def score_relevance_logits(model, tokenizer, prompts, batch_size=4):
    """
    Score relevance using token log-probabilities — continuous 0-to-1 score.
    Inputs:
        model      : 4-bit Mistral model
        tokenizer  : Mistral tokenizer
        prompts    : list of formatted prompt strings
        batch_size : prompts per GPU batch (reduce if OOM)

    Outputs:
        list of float scores in [0, 1], one per prompt
    """
    scores = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="  Scoring", leave=False):
        batch = prompts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)   # forward pass only, no generation

        for j in range(len(batch)):
            # Last non-padded token → its logit predicts the first answer token
            last_real_pos = inputs["attention_mask"][j].sum().item() - 1
            logits_at_pos = outputs.logits[j, last_real_pos, :].float()

            log_probs = torch.log_softmax(logits_at_pos, dim=-1)
            rel_lp    = log_probs[RELEVANT_TOKEN_ID].item()
            not_lp    = log_probs[NOT_TOKEN_ID].item()

            # Softmax over {relevant, not} → probability that answer is "relevant"
            rel_exp   = torch.exp(torch.tensor(rel_lp))
            not_exp   = torch.exp(torch.tensor(not_lp))
            rel_score = (rel_exp / (rel_exp + not_exp)).item()
            scores.append(rel_score)

    return scores



_test_texts = [
    "I feel so empty and hopeless, nothing makes me happy anymore.",  # should score HIGH
    "Just bought a new coffee machine, absolutely love it.",            # should score LOW
]
_test_prompts = prepare_few_shot_prompts(
    texts=_test_texts, symptom_desc=BDI_SYMPTOMS["1"],
    prompt_template=few_shot_prompt, tokenizer=mistral_tokenizer, qid="1"
)
_test_scores = score_relevance_logits(mistral_model, mistral_tokenizer, _test_prompts, batch_size=2)
print("Sanity check scores (Query 1 — Sadness):")
for txt, sc in zip(_test_texts, _test_scores):
    print(f"  {sc:.4f}  |  {txt[:70]}")

Token for ' relevant'  : id=  9366  decoded='relevant'
Token for ' not'       : id=  1227  decoded='not'



Sanity check scores (Query 1 — Sadness):
  1.0000  |  I feel so empty and hopeless, nothing makes me happy anymore.
  0.0019  |  Just bought a new coffee machine, absolutely love it.


## Full Mistral Few-Shot Reranker Pipeline



In [ ]:
def run_mistral_reranker(
    cursor,
    queries     : dict,
    output_path : str,
    run_name    : str,
    model,
    tokenizer,
    bm25_top_k  : int = 1000,
    batch_size  : int = 4,
    top_k       : int = 1000,
) -> dict:
    """
    Full few-shot pipeline:
      1. BM25 retrieves bm25_top_k candidates per query from the 17M-post index.
      2. prepare_few_shot_prompts builds a Mistral instruction prompt per candidate
         using hardcoded examples (no data leakage).
      3. score_relevance_logits runs a forward pass and returns a continuous
         relevance probability per candidate.
      4. Candidates are ranked by that probability and written in TREC format.

    Output: stats dict {qid: number_of_docs_written}
    """
    stats = {}

    with open(output_path, "w") as f_out:
        for qid, raw_query in queries.items():
            print(f"\n  Query {qid} — {BDI_SYMPTOMS[qid][:50]}...")

            # ── Step 1: BM25 retrieval ──────────────────────────────────────
            fts5_query = format_fts5_query(raw_query)
            try:
                cursor.execute("""
                    SELECT doc_id, content, -bm25(posts) AS bm25_score
                    FROM   posts
                    WHERE  posts MATCH ?
                    ORDER  BY bm25_score DESC
                    LIMIT  ?
                """, (fts5_query, bm25_top_k))
                candidates = cursor.fetchall()
            except Exception as e:
                print(f"  BM25 failed: {e}")
                stats[qid] = 0
                continue

            if not candidates:
                print("  No BM25 candidates.")
                stats[qid] = 0
                continue

            doc_ids = [r[0] for r in candidates]
            texts   = [r[1] for r in candidates]
            print(f"  BM25 retrieved {len(candidates):,} candidates")

            # ── Step 2: Format prompts (hardcoded examples, no qrels) ───────
            prompts = prepare_few_shot_prompts(
                texts           = texts,
                symptom_desc    = BDI_SYMPTOMS[qid],
                prompt_template = few_shot_prompt,
                tokenizer       = tokenizer,
                qid             = qid,
            )

            valid = [(d, p) for d, p in zip(doc_ids, prompts) if p is not None]
            if not valid:
                print("  All prompts failed to format.")
                stats[qid] = 0
                continue

            valid_doc_ids = [v[0] for v in valid]
            valid_prompts = [v[1] for v in valid]

            # ── Step 3: Score with logits (continuous, not 0/0.5/1) ─────────
            print(f"  Scoring {len(valid_prompts):,} prompts with Mistral (batch={batch_size})...")
            scores = score_relevance_logits(model, tokenizer, valid_prompts, batch_size=batch_size)

            # ── Step 4: Rank and write TREC output ───────────────────────────
            ranked = sorted(zip(valid_doc_ids, scores), key=lambda x: x[1], reverse=True)
            for rank, (doc_id, score) in enumerate(ranked[:top_k], start=1):
                f_out.write(f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_name}\n")

            stats[qid] = len(ranked)
            hi = sum(1 for _, s in ranked if s >= 0.7)
            lo = sum(1 for _, s in ranked if s <  0.3)
            print(f"  Done — {len(ranked):,} docs | score>=0.7: {hi} | score<0.3: {lo}")

    return stats


print("run_mistral_reranker ready.")

run_mistral_reranker ready.


## Run Mistral Reranker

The code runs a few-shot Mistral-based reranker over BM25 candidates for BDI queries:

**Inputs:**
* BDI_QUERIES – the set of depressive symptom queries.
* mistral_model and mistral_tokenizer – the 4-bit Mistral LLM and tokenizer.
* bm25_top_k – number of BM25 candidates per query.
* top_k – number of final reranked documents to keep.
* batch_size – prompts processed per batch on GPU.

**Process:**
* Retrieves BM25 candidates from the database.
* Prepares few-shot prompts for each query and candidate.
* Uses score_relevance_logits to compute a probability score that each document is relevant.
* Ranks candidates by these Mistral-generated scores and writes the top results to a TREC-style run file.

In [ ]:
mistral_stats = run_mistral_reranker(
    cursor      = c,
    queries     = BDI_QUERIES,
    output_path = RUN_MISTRAL,
    run_name    = "Mistral_FewShot",
    model       = mistral_model,
    tokenizer   = mistral_tokenizer,
    bm25_top_k  = 1000,
    batch_size  = 4,
    top_k       = 1000,
)

print("\n=== Mistral Run Summary ===")
for qid, count in mistral_stats.items():
    flag = "" if count < 100 else "  "
    print(f"  {flag} Query {qid:>2} : {count:,} docs written")
print(f"\nRun file saved to: {RUN_MISTRAL}")


  Query 1 — Sadness — feeling sad, empty, or hopeless...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 280 | score<0.3: 682

  Query 2 — Pessimism — feeling discouraged about the future...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 235 | score<0.3: 736

  Query 3 — Past Failure — feeling like a failure...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 245 | score<0.3: 685

  Query 4 — Loss of Pleasure — not getting pleasure from thing...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 341 | score<0.3: 627

  Query 5 — Guilty Feelings — feeling guilty...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


...

  Done — 1,000 docs | score>=0.7: 342 | score<0.3: 613

  Query 18 — Appetite Changes — eating too much or too little...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 470 | score<0.3: 455

  Query 19 — Concentration Difficulty — trouble concentrating...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 166 | score<0.3: 805

  Query 20 — Tiredness — feeling tired all the time...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 370 | score<0.3: 596

  Query 21 — Loss of Interest in Sex — less interested in sex...
  BM25 retrieved 1,000 candidates
  Scoring 1,000 prompts with Mistral (batch=4)...


  Done — 1,000 docs | score>=0.7: 272 | score<0.3: 684

=== Mistral Run Summary ===
     Query  1 : 1,000 docs written
     Query  2 : 1,000 docs written
     Query  3 : 1,000 docs written
     Query  4 : 1,000 docs written
     Query  5 : 1,000 docs written
     Query  6 : 1,000 docs written
     Query  7 : 1,000 docs written
     Query  8 : 1,000 docs written
     Query  9 : 1,000 docs written
     Query 10 : 1,000 docs written
     Query 11 : 1,000 docs written
     Query 12 : 1,000 docs written
     Query 13 : 1,000 docs written
     Query 14 : 1,000 docs written
     Query 15 : 1,000 docs written
     Query 16 : 1,000 docs written
     Query 17 : 1,000 docs written
     Query 18 : 1,000 docs written
     Query 19 : 1,000 docs written
     Query 20 : 1,000 docs written
     Query 21 : 1,000 docs written

Run file saved to: /content/drive/MyDrive/nlpwork/run_mistral_llm.txt


## Final Evaluation — All 4 Systems

The code evaluates all retrieval and reranking systems using TREC metrics:

**Runs included:**
* BM25 Baseline – standard BM25 retrieval.
* Rerank MiniLM – BM25 candidates reranked with MiniLM embeddings.
* Hybrid MiniLM – combination of BM25 + MiniLM scores.
* Mistral LLM – few-shot Mistral-based reranking.

**Process per system:**
* Checks if the run file exists.
Uses trec_eval to compute evaluation metrics:
* MAP – Mean Average Precision
* nDCG@10 – normalized Discounted Cumulative Gain at rank 10
* P@10 – Precision at 10


In [ ]:
print("\n Evaluating all systems (BM25 / Rerank / Hybrid / Mistral):")

for run_name, run_path in {
    "BM25 Baseline" : RUN_BM25,
    "Rerank MiniLM" : RUN_RERANK,
    "Hybrid MiniLM" : RUN_HYBRID,
    "Mistral LLM"   : RUN_MISTRAL,
}.items():
    if not os.path.exists(run_path):
        print(f"\n  {run_name} — run file not found: {run_path}")
        continue
    print(f"\n{'='*50}")
    print(f"  {run_name}")
    print(f"{'='*50}")
    !./trec_eval/trec_eval -J -m map -m ndcg_cut.10 -m P.10 {QRELS_TREC} {run_path}


📊 Evaluating all systems (BM25 / Rerank / Hybrid / Mistral):

  BM25 Baseline
map                   	all	0.0515
P_10                  	all	0.5905
ndcg_cut_10           	all	0.6240

  Rerank MiniLM
map                   	all	0.0537
P_10                  	all	0.6238
ndcg_cut_10           	all	0.6384

  Hybrid MiniLM
map                   	all	0.0535
P_10                  	all	0.6238
ndcg_cut_10           	all	0.6394

  Mistral LLM
map                   	all	0.0572
P_10                  	all	0.6905
ndcg_cut_10           	all	0.7097


## Top-5 Retrieved Sentences — All Approaches × All 21 BDI Queries

For each of the 4 systems, shows the top-5 ranked sentences for every BDI query. This lets visually inspect whether the retrieval makes sense, do the sentences actually match the symptom?

In [ ]:
def load_run_file(path):
    """
    Parse a TREC run file into:
        {qid: [(doc_id, rank, score), ...]} sorted by rank ascending
    """
    runs = {}
    if not os.path.exists(path):
        return runs
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            qid, _, doc_id, rank, score, _ = parts[0], parts[1], parts[2], int(parts[3]), float(parts[4]), parts[5]
            runs.setdefault(qid, []).append((doc_id, rank, score))
    for qid in runs:
        runs[qid].sort(key=lambda x: x[1])
    return runs


def fetch_texts(cursor, doc_ids):
    """
    Retrieve content for a list of doc_ids from posts_lookup.
    Returns {doc_id: text}
    """
    if not doc_ids:
        return {}
    placeholders = ",".join("?" * len(doc_ids))
    cursor.execute(
        f"SELECT doc_id, content FROM posts_lookup WHERE doc_id IN ({placeholders})",
        doc_ids
    )
    return {row[0]: row[1] for row in cursor.fetchall()}


SYSTEMS = {
    "BM25 Baseline" : RUN_BM25,
    "Rerank MiniLM" : RUN_RERANK,
    "Hybrid MiniLM" : RUN_HYBRID,
    "Mistral LLM"   : RUN_MISTRAL,
}

# Pre-load all run files
all_runs = {name: load_run_file(path) for name, path in SYSTEMS.items()}

TOP_N = 5

print(f"Showing top-{TOP_N} sentences per system per BDI query")
print(f"Total queries: {len(BDI_QUERIES)}")
print(f"Total systems: {len(SYSTEMS)}")
print("=" * 80)

for qid in sorted(BDI_QUERIES.keys(), key=lambda x: int(x)):
    symptom = BDI_SYMPTOMS[qid]
    print(f"\n{'#' * 80}")
    print(f"  BDI Query {qid:>2}  |  {symptom}")
    print(f"{'#' * 80}")

    for sys_name, run_data in all_runs.items():

        print(f"\n  ── {sys_name} ──")

        if qid not in run_data or not run_data[qid]:
            print("    [no results]")
            continue

        top_entries = run_data[qid][:TOP_N]
        top_doc_ids = [e[0] for e in top_entries]

        # Fetch texts from DB
        text_map = fetch_texts(c, top_doc_ids)

        for i, (doc_id, rank, score) in enumerate(top_entries, start=1):
            raw_text = text_map.get(doc_id, "[text not found in index]")
            # Truncate for display — 200 chars is enough to judge relevance
            display  = raw_text[:200].strip()
            if len(raw_text) > 200:
                display += "..."
            print(f"    {i}. [score={score:.4f}] {doc_id}")
            print(f"       {display}")
            print()

print("\n" + "=" * 80)
print("Top-5 display complete.")

Showing top-5 sentences per system per BDI query
Total queries: 21
Total systems: 4

################################################################################
  BDI Query  1  |  Sadness — feeling sad, empty, or hopeless
################################################################################

  ── BM25 Baseline ──
    1. [score=23.4590] pWCgOz_4913_2
       you must be so miserable and unhappy to do this kind of stuff. it's sad that people like you are so miserable they fed the need to do this

    2. [score=22.9449] xvE2fn_4499_4
       you're gloomy. i knew i can talk to you, daria. you're always miserable."

    3. [score=20.7199] bNwvw6_757_6
       if i feel okay tonight i'm drowning my sorrows in a glass of wine. sad sad sad.

    4. [score=20.4417] OmFEBt_84_0
       grey gloomy day looking down woodward ave in detroit

    5. [score=20.4279] h2dHah_506_0
       i'm sad, and unhappy


  ── Rerank MiniLM ──
    1. [score=0.6508] YkN48U_1998_2
       sad is when the